# CSE 590A — Post-Discharge Follow-Up Agent · Colab Runner

End-to-end GPU run of the systems study. Three configs:

| Config | What | Prefix cache |
|--------|------|--------------|
| **A** | single-shot baseline (one LLM call/patient) | n/a |
| **B** | multi-agent (3 workers + assembler + reflection) | **OFF** |
| **C** | multi-agent (identical code to B) | **ON** |


## 0. Confirm the GPU (expect a T4)

In [ ]:
!nvidia-smi

Fri Jun 12 03:38:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             31W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install dependencies

In [ ]:
!pip install -q vllm openai requests matplotlib numpy huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.0 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.3.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 

## 2. Clone the project

In [ ]:
import os
if not os.path.isdir('cse590a-project'):
    !git clone https://github.com/sais14/cse590a-project.git
%cd cse590a-project
!git pull --ff-only

/content/cse590a-project
Already up to date.


## 3. Hugging Face login (gated model)

In [ ]:
from huggingface_hub import login
login()  # paste an HF token with read access to meta-llama/Llama-3.2-3B-Instruct

## 4. Build the data pipeline

### Why the data isn't in the repo

To keep the GitHub repo small, the large, regenerable data artifacts are
git-ignored and are *not* part of the clone. Only the code plus the small
derived files (the 30 discharge notes, `ground_truth.json`, `notes_manifest.json`)
are committed. What's excluded, and where it comes from:

| Not in repo | Size | What it is | Regenerated by |
|-------------|------|------------|----------------|
| `synthea_sample.zip` | ~9 MB | Synthea synthetic-patient sample (CSV, apr2020) | downloaded from the public Synthea mirror |
| `synthea_data/csv/*.csv` | ~79 MB | the unzipped raw Synthea CSVs | unzip of the above |
| `data/synthea.db` | ~96 MB | normalized SQLite history DB (8 tables, 736-patient cohort) | `build_db.py` |

`data/synthea.db` alone exceeds GitHub's 100 MB file limit, so committing it
isn't an option — but everything is **fully reproducible** from public data.

### One command rebuilds everything

`setup.sh`: it downloads + unzips the Synthea sample, runs
`build_db.py` to build the SQLite DB, then runs `generate_notes.py`. It **skips**
any step whose output already exists (e.g. it won't re-download if
`data/synthea.db` is already there). Just run the cell below.

In [ ]:
!bash setup.sh

[skip] data/synthea.db already exists — skipping download + DB build.
[skip] data/notes already populated — skipping note generation.
[done] data pipeline ready.


<details>
<summary><b>Manual download </b></summary>

`setup.sh` above already does all of this. These are the equivalent manual steps,

```bash
# 1. Download + unzip the public Synthea sample (CSV, apr2020)
curl -fSL \
  https://raw.githubusercontent.com/synthetichealth/synthea-sample-data/main/downloads/synthea_sample_data_csv_apr2020.zip \
  -o synthea_sample.zip
unzip -oq synthea_sample.zip -d synthea_data    # -> synthea_data/csv/*.csv

# 2. Build the normalized SQLite history DB (-> data/synthea.db)
python build_db.py

# 3. Generate discharge notes + ground truth (already committed, but reproducible)
python generate_notes.py

# Inspect the DB
sqlite3 data/synthea.db ".tables"
sqlite3 data/synthea.db "SELECT COUNT(*) FROM eligible_patients;"   # -> 736
```
</details>

## 5. vLLM server helpers

Launch vLLM in the background, poll `/health` until ready, run benchmarks, then
stop it cleanly before relaunching with a different cache setting.

In [ ]:
import subprocess, time, requests, signal

MODEL = "meta-llama/Llama-3.2-3B-Instruct"
SERVER_LOG = "/content/vllm.log"
_proc = None

def launch_vllm(prefix_cache: bool):
    global _proc
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--dtype", "float16",
        "--max-model-len", "8192",
        "--port", "8000",
        "--gpu-memory-utilization", "0.9",
        "--guided-decoding-backend", "lm-format-enforcer",
    ]
    if prefix_cache:
        cmd.append("--enable-prefix-caching")
    log = open(SERVER_LOG, "w")
    _proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    print(f"launched vLLM (prefix_cache={prefix_cache}), pid={_proc.pid}; logging -> {SERVER_LOG}")
    return _proc

def wait_for_server(timeout=900):
    start = time.time()
    url = "http://localhost:8000/health"
    while time.time() - start < timeout:
        if _proc is not None and _proc.poll() is not None:
            raise RuntimeError(f"vLLM exited early (code {_proc.returncode}); see {SERVER_LOG}")
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print(f"server ready in {time.time()-start:.0f}s")
                return
        except Exception:
            pass
        time.sleep(3)
    raise TimeoutError(f"server not ready after {timeout}s; see {SERVER_LOG}")

def stop_vllm():
    global _proc
    if _proc is None:
        return
    _proc.send_signal(signal.SIGINT)
    try:
        _proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        _proc.kill()
    print("server stopped")
    _proc = None

## 6. Configs A + B — prefix cache **OFF**

In [ ]:
launch_vllm(prefix_cache=False)
wait_for_server()

launched vLLM (prefix_cache=False), pid=15842; logging -> /content/vllm.log
server ready in 123s


In [ ]:
!python benchmark.py --config A
!python benchmark.py --config B

Using AsyncOpenAI at http://localhost:8000/v1 (metrics: http://localhost:8000/metrics).

=== Config A over 30 patients ===
  [A] note_00 tokens=  825 latency= 9.506s calls=1 reflect=0
  [A] note_01 tokens= 1717 latency=27.028s calls=1 reflect=0
  [A] note_02 tokens= 2147 latency=47.022s calls=1 reflect=0
  [A] note_03 tokens=  888 latency=12.796s calls=1 reflect=0
  [A] note_04 tokens= 1119 latency=14.163s calls=1 reflect=0
  [A] note_05 tokens= 1089 latency=13.546s calls=1 reflect=0
  [A] note_06 tokens=  871 latency=17.715s calls=1 reflect=0
  [A] note_07 tokens= 1153 latency=20.718s calls=1 reflect=0
  [A] note_08 tokens=  400 latency=13.334s calls=1 reflect=0
  [A] note_09 tokens= 1170 latency=15.740s calls=1 reflect=0
  [A] note_10 tokens= 1859 latency=44.639s calls=1 reflect=0
  [A] note_11 tokens=  597 latency=10.869s calls=1 reflect=0
  [A] note_12 tokens=  934 latency=27.824s calls=1 reflect=0
  [A] note_13 tokens=  887 latency=22.947s calls=1 reflect=0
  [A] note_14 tokens=  

In [ ]:
stop_vllm()

server stopped


## 7. Config C — prefix cache **ON** (restart server first)

In [ ]:
launch_vllm(prefix_cache=True)
wait_for_server()

launched vLLM (prefix_cache=True), pid=22710; logging -> /content/vllm.log
server ready in 126s


In [ ]:
!python benchmark.py --config C

Using AsyncOpenAI at http://localhost:8000/v1 (metrics: http://localhost:8000/metrics).

=== Config C over 30 patients ===
  [C] note_00 tokens=  825 latency=18.719s calls=3 reflect=0
  [C] note_01 tokens= 1717 latency=15.688s calls=3 reflect=0
  [C] note_02 tokens= 2147 latency=32.630s calls=3 reflect=0
  [C] note_03 tokens=  888 latency= 8.656s calls=3 reflect=0
  [C] note_04 tokens= 1119 latency= 9.167s calls=3 reflect=0
  [C] note_05 tokens= 1089 latency= 8.158s calls=3 reflect=0
  [C] note_06 tokens=  871 latency=10.069s calls=3 reflect=0
  [C] note_07 tokens= 1153 latency=11.589s calls=3 reflect=0
  [C] note_08 tokens=  400 latency= 6.935s calls=3 reflect=0
  [C] note_09 tokens= 1170 latency= 7.972s calls=3 reflect=0
  [C] note_10 tokens= 1859 latency=19.808s calls=3 reflect=0
  [C] note_11 tokens=  597 latency= 6.938s calls=3 reflect=0
  [C] note_12 tokens=  934 latency=10.826s calls=3 reflect=0
  [C] note_13 tokens=  887 latency= 8.747s calls=3 reflect=0
  [C] note_14 tokens=  

In [ ]:
stop_vllm()

server stopped


## 8. Evaluate + plots

Computes F1 metrics and writes `results/evaluation.json` plus the latency-vs-length
and Pareto PDFs under `results/plots/`.

In [ ]:
!python evaluate.py

Wrote results/evaluation.json (configs: ['A', 'B', 'C'])

=== Worker x Config table ===
metric                         A         B         C
medication F1              0.927     0.107     0.107
problem F1                 0.854     0.939     0.928
followup approp.           0.862     0.325     0.325
mean worker F1             0.881     0.457     0.453
schema validity            0.933     0.458     0.458
Wrote results/plots/latency_vs_length.pdf
Wrote results/plots/pareto.pdf


In [ ]:
import json, glob
print(json.dumps(json.load(open("results/evaluation.json")), indent=2))
print("\nplots:")
for p in sorted(glob.glob("results/plots/*.pdf")):
    print(" ", p)

{
  "A": {
    "medication": {
      "precision": 0.9285714285714286,
      "recall": 0.9258241758241759,
      "f1": 0.9271428571428572,
      "n_evaluated": 28
    },
    "problem": {
      "precision": 0.92,
      "recall": 0.8212301587301588,
      "f1": 0.8541032393855982,
      "n_evaluated": 30
    },
    "followup": {
      "appropriateness": 0.8618055555555556,
      "n_evaluated": 24
    },
    "json_validity_rate": 1.0,
    "schema_validity_rate": 0.9333333333333333,
    "mean_worker_f1": 0.881017217361337
  },
  "B": {
    "medication": {
      "precision": 0.10714285714285714,
      "recall": 0.10714285714285714,
      "f1": 0.10714285714285714,
      "n_evaluated": 28
    },
    "problem": {
      "precision": 0.985,
      "recall": 0.9081349206349206,
      "f1": 0.9389672481626505,
      "n_evaluated": 30
    },
    "followup": {
      "appropriateness": 0.325,
      "n_evaluated": 24
    },
    "json_validity_rate": 0.4694444444444444,
    "schema_validity_rate": 0.458